# Week 2: Advanced Functions & Docstrings — PHASE 2: The Decomposition Recipe

*Core Mastery: "I can write well-documented functions with flexible argument patterns"*

*Computer Programming II | 5 Hours | Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. Define functions with **default argument** values
2. Call functions using **keyword arguments** for clarity
3. Explain and avoid the **mutable default argument trap**
4. Write **Google-style docstrings** (one-line and multi-line)
5. Use `help()` and `__doc__` to inspect function documentation
6. Build a small **function library** in a notebook
7. Write basic **assertions** to test function correctness
8. Combine defaults, keyword arguments, and docstrings in professional functions
9. Distinguish between positional and keyword-only arguments
10. Apply the testing mindset: write the test *before* the function body

## 🎯 Core Mastery Connection

Last week you reviewed the CP1 basics and learned the refactoring mindset. This week we focus on the **function** as the fundamental unit of clean code. A well-written function has a clear *contract*: it documents what it expects, what it returns, and handles edge cases gracefully. Mastering default arguments, keyword arguments, and docstrings transforms your functions from "scripts with a name" into reusable, professional building blocks.

---
## Part 1: Functions Recap — def, Parameters, Return

A quick refresher on function basics:

```python
def function_name(parameter1, parameter2):
    """What the function does."""
    result = ...  # computation
    return result
```

**Key rules:**
- `def` introduces the function
- Parameters are local variables
- `return` sends back a value (default is `None` if omitted)
- The body is indented one level

In [ ]:
# Basic function — Zeynep's area calculator
def rectangle_area(width, height):
    """Calculate the area of a rectangle."""
    return width * height

def triangle_area(base, height):
    """Calculate the area of a triangle."""
    return 0.5 * base * height

# Usage
print(f"Rectangle 5×3 : {rectangle_area(5, 3)}")
print(f"Triangle  5×3 : {triangle_area(5, 3)}")

**Figure 1.1** — Two simple geometry functions.

In [ ]:
# Functions with no return → return None
def print_header(title):
    """Print a formatted section header."""
    print("=" * 40)
    print(f"  {title.upper()}")
    print("=" * 40)

result = print_header("sensor report")
print(f"\nReturn value: {result}")  # None

**Figure 1.2** — A function without `return` implicitly returns `None`.

In [ ]:
# Multiple return values (tuple unpacking)
def min_max(values):
    """Return the minimum and maximum of a list."""
    return min(values), max(values)

lo, hi = min_max([34, 12, 78, 56, 23])
print(f"Min: {lo}, Max: {hi}")

**Figure 1.3** — Returning multiple values via tuple unpacking.

---
## Part 2: Default Arguments — When and Why

A **default argument** gives a parameter a fallback value. Callers can omit it or override it.

```python
def greet(name, greeting="Merhaba"):
    return f"{greeting}, {name}!"
```

**Rules:**
- Default parameters must come *after* non-default parameters
- Defaults are evaluated *once* at function definition time (important for Part 4!)
- Use defaults for the *most common* use case

In [ ]:
# Default arguments — Ahmet's unit converter
def convert_temperature(celsius, to_unit="F"):
    """Convert Celsius to Fahrenheit (default) or Kelvin."""
    if to_unit.upper() == "F":
        return celsius * 9 / 5 + 32
    elif to_unit.upper() == "K":
        return celsius + 273.15
    else:
        raise ValueError(f"Unknown unit: {to_unit}")

# Using the default
print(f"100°C = {convert_temperature(100)}°F")

# Overriding the default
print(f"100°C = {convert_temperature(100, 'K')} K")

**Figure 2.1** — A function with a sensible default for the output unit.

In [ ]:
# Multiple defaults — sensor configuration
def read_sensor(sensor_id, samples=10, interval_ms=100, verbose=False):
    """Simulate reading a sensor with configurable parameters."""
    if verbose:
        print(f"Reading sensor {sensor_id}: {samples} samples, {interval_ms}ms interval")
    # Simulate readings
    import random
    readings = [round(random.gauss(25, 2), 1) for _ in range(samples)]
    return readings

# All defaults
data1 = read_sensor("DHT22")
print(f"Default (10 samples): {data1}")

# Override just samples
data2 = read_sensor("DHT22", samples=3)
print(f"3 samples: {data2}")

# Override samples and verbose
data3 = read_sensor("DHT22", samples=3, verbose=True)

**Figure 2.2** — Multiple default parameters let callers customize only what they need.

In [ ]:
# ❌ Common mistake: default before non-default
# def bad_func(x=10, y):   # SyntaxError!
#     return x + y

# ✅ Correct order
def good_func(y, x=10):
    return x + y

print(good_func(5))       # y=5, x=10 → 15
print(good_func(5, 20))   # y=5, x=20 → 25

**Figure 2.3** — Default parameters must follow non-default parameters.

---
## Part 3: Keyword Arguments — Calling with Names

When calling a function, you can specify arguments by name:

```python
result = convert_temperature(celsius=100, to_unit="K")
```

**Benefits:**
- Clearer code at the call site
- Order does not matter when using keyword arguments
- Self-documenting: the reader sees *what* each value means

In [ ]:
# Without keyword args — what do these numbers mean?
# ❌ Unclear
result1 = read_sensor("BMP280", 5, 200, True)

# ✅ Crystal clear with keyword arguments
result2 = read_sensor("BMP280", samples=5, interval_ms=200, verbose=True)

**Figure 3.1** — Keyword arguments make function calls self-documenting.

In [ ]:
# Keyword arguments can be in any order
def create_resistor(value, tolerance=5, power_rating=0.25, unit="ohm"):
    """Create a resistor specification dictionary."""
    return {
        "value": value,
        "tolerance_pct": tolerance,
        "power_w": power_rating,
        "unit": unit
    }

# All of these are equivalent:
r1 = create_resistor(470, tolerance=1, power_rating=0.5)
r2 = create_resistor(470, power_rating=0.5, tolerance=1)

print(f"r1: {r1}")
print(f"r2: {r2}")
print(f"Same? {r1 == r2}")

**Figure 3.2** — Keyword argument order does not matter.

In [ ]:
# Mixing positional and keyword
def beam_deflection(force, length, E, I):
    """Calculate max deflection of a simply supported beam (center load)."""
    return (force * length**3) / (48 * E * I)

# First arg positional, rest as keywords for clarity
deflection = beam_deflection(
    1000,               # force in N
    length=6.0,         # meters
    E=200e9,            # Pa (steel)
    I=8.33e-6           # m^4
)
print(f"Max deflection: {deflection*1000:.3f} mm")

**Figure 3.3** — Mixing positional and keyword arguments for engineering calculations.

---
## Part 4: The Mutable Default Trap

One of Python’s most famous gotchas:

> **Default values are evaluated once, when the function is defined, not each time it is called.**

This is fine for immutable defaults (`int`, `str`, `tuple`), but **dangerous for mutable defaults** (`list`, `dict`, `set`) because the same object is shared across all calls.

### The Pattern

```python
# ❌ WRONG — shared list
def add_item(item, items=[]):
    items.append(item)
    return items

# ✅ CORRECT — use None sentinel
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items
```

In [ ]:
# ❌ The mutable default trap in action
def add_reading(value, readings=[]):
    """Append a reading to the list (BUGGY!)."""
    readings.append(value)
    return readings

# Watch what happens:
print("Call 1:", add_reading(10))     # [10] — looks fine
print("Call 2:", add_reading(20))     # [10, 20] — WAIT, where did 10 come from?!
print("Call 3:", add_reading(30))     # [10, 20, 30] — the list persists!

**Figure 4.1** — The mutable default trap: the list accumulates across calls.

In [ ]:
# ✅ The fix: use None as sentinel
def add_reading_fixed(value, readings=None):
    """Append a reading to the list (CORRECT)."""
    if readings is None:
        readings = []
    readings.append(value)
    return readings

print("Call 1:", add_reading_fixed(10))    # [10]
print("Call 2:", add_reading_fixed(20))    # [20] — fresh list each time!
print("Call 3:", add_reading_fixed(30))    # [30]

# Can still pass an existing list
existing = [1, 2, 3]
print("With existing:", add_reading_fixed(4, existing))  # [1, 2, 3, 4]

**Figure 4.2** — The `None` sentinel pattern creates a fresh list each call.

In [ ]:
# Same trap with dictionaries
def add_config(key, value, config=None):
    """Add a key-value pair to a config dictionary."""
    if config is None:
        config = {}
    config[key] = value
    return config

c1 = add_config("sensor", "DHT22")
c2 = add_config("rate", 10)
print(f"c1: {c1}")   # {'sensor': 'DHT22'}
print(f"c2: {c2}")   # {'rate': 10} — independent!

**Figure 4.3** — The `None` sentinel works the same way for `dict` defaults.

---
## Part 5: Google-Style Docstrings — One-Line and Full Format

A **docstring** is a string literal that appears as the first statement in a function. Python stores it in `__doc__` and `help()` displays it.

### One-line docstring
For very simple functions:
```python
def square(x):
    """Return the square of x."""
    return x ** 2
```

### Google-style multi-line docstring
```python
def voltage_divider(vin, r1, r2):
    """Calculate the output voltage of a resistive voltage divider.

    Args:
        vin: Input voltage in volts.
        r1: Top resistor in ohms.
        r2: Bottom resistor in ohms.

    Returns:
        Output voltage in volts.

    Raises:
        ValueError: If r1 + r2 is zero.
    """
```

The Google style uses `Args:`, `Returns:`, `Raises:` sections.

In [ ]:
# One-line docstring
def celsius_to_kelvin(c):
    """Convert Celsius to Kelvin."""
    return c + 273.15

print(celsius_to_kelvin(25))
print(celsius_to_kelvin.__doc__)

**Figure 5.1** — One-line docstring for a trivial function.

In [ ]:
# Full Google-style docstring — Fatma's beam calculator
def cantilever_deflection(force, length, E, I):
    """Calculate the max deflection of a cantilever beam with end load.

    Uses the formula: delta = (F * L^3) / (3 * E * I)

    Args:
        force: Applied force at the free end in Newtons.
        length: Beam length in meters.
        E: Young's modulus in Pascals (e.g., 200e9 for steel).
        I: Second moment of area in m^4.

    Returns:
        Maximum deflection in meters (positive = downward).

    Raises:
        ValueError: If E or I is zero or negative.
    """
    if E <= 0 or I <= 0:
        raise ValueError("E and I must be positive")
    return (force * length**3) / (3 * E * I)

# Test it
d = cantilever_deflection(500, 3.0, 200e9, 4.16e-6)
print(f"Deflection: {d*1000:.2f} mm")

**Figure 5.2** — A full Google-style docstring with Args, Returns, and Raises.

In [ ]:
# Docstring with default arguments documented
def format_reading(value, unit="°C", decimals=1, prefix=""):
    """Format a sensor reading as a display string.

    Args:
        value: The numeric reading.
        unit: Unit label to append. Defaults to '°C'.
        decimals: Decimal places for formatting. Defaults to 1.
        prefix: Optional prefix text. Defaults to ''.

    Returns:
        Formatted string like '23.5°C' or 'Temp: 23.5°C'.
    """
    formatted = f"{value:.{decimals}f}{unit}"
    if prefix:
        formatted = f"{prefix}: {formatted}"
    return formatted

print(format_reading(23.456))
print(format_reading(1013.25, unit=" hPa", decimals=0, prefix="Pressure"))

**Figure 5.3** — Documenting default argument values in the docstring.

---
## Part 6: The help() Function and __doc__

Python’s built-in `help()` function reads the docstring and displays it in a formatted way. You can also access the raw docstring via `__doc__`.

- `help(function_name)` — formatted output
- `function_name.__doc__` — raw string

This works for your own functions *and* built-in functions.

In [ ]:
# help() on our own function
help(cantilever_deflection)

**Figure 6.1** — `help()` displays the full docstring.

In [ ]:
# __doc__ attribute
print(format_reading.__doc__)

**Figure 6.2** — Accessing the raw docstring via `__doc__`.

In [ ]:
# help() on built-in functions
help(len)

**Figure 6.3** — `help()` works on built-in functions too.

In [ ]:
# Why docstrings matter: undocumented vs documented
def mystery(a, b, c):
    return (-b + (b**2 - 4*a*c)**0.5) / (2*a)

def quadratic_root(a, b, c):
    """Return the positive root of ax^2 + bx + c = 0 using the quadratic formula.

    Args:
        a: Coefficient of x^2 (must not be zero).
        b: Coefficient of x.
        c: Constant term.

    Returns:
        The root (-b + sqrt(b^2 - 4ac)) / (2a).
    """
    return (-b + (b**2 - 4*a*c)**0.5) / (2*a)

# Which would you rather find in someone's code?
print("mystery(1, -5, 6)        =", mystery(1, -5, 6))
print("quadratic_root(1, -5, 6) =", quadratic_root(1, -5, 6))

**Figure 6.4** — Docstrings transform cryptic code into self-documenting code.

---
## Part 7: Building a Function Library

A **function library** is a collection of related, well-documented functions that you can reuse across projects. In a notebook, this means a group of cells near the top (after CONFIG) containing utility functions.

### Library Design Principles

| Principle | Description |
|-----------|-------------|
| **Single purpose** | Each function does one thing |
| **Consistent style** | Same naming, same docstring format |
| **No side effects** | Functions return values, not print them |
| **Tested** | Each function has at least one assertion |

In [ ]:
# ═══════════════════════════════════════════
# FUNCTION LIBRARY — Engineering Calculations
# ═══════════════════════════════════════════

def ohms_law_current(voltage, resistance):
    """Calculate current using Ohm's law: I = V / R.

    Args:
        voltage: Voltage in volts.
        resistance: Resistance in ohms (must be > 0).

    Returns:
        Current in amperes.
    """
    if resistance <= 0:
        raise ValueError("Resistance must be positive")
    return voltage / resistance

def power_dissipation(voltage, current):
    """Calculate electrical power: P = V * I.

    Args:
        voltage: Voltage in volts.
        current: Current in amperes.

    Returns:
        Power in watts.
    """
    return voltage * current

def resistors_in_series(*resistors):
    """Calculate total resistance of resistors in series.

    Args:
        *resistors: One or more resistance values in ohms.

    Returns:
        Total resistance in ohms.
    """
    return sum(resistors)

def resistors_in_parallel(*resistors):
    """Calculate total resistance of resistors in parallel.

    Args:
        *resistors: One or more resistance values in ohms (all > 0).

    Returns:
        Total resistance in ohms.
    """
    if any(r <= 0 for r in resistors):
        raise ValueError("All resistances must be positive")
    return 1 / sum(1/r for r in resistors)

print("Library loaded: ohms_law_current, power_dissipation, resistors_in_series, resistors_in_parallel")

**Figure 7.1** — A small engineering function library with consistent style.

In [ ]:
# Using the library — Ali's circuit analysis
v_supply = 12.0  # V
r1, r2, r3 = 1000, 2200, 3300  # ohms

# Series circuit
r_series = resistors_in_series(r1, r2, r3)
i_series = ohms_law_current(v_supply, r_series)
p_series = power_dissipation(v_supply, i_series)

print(f"Series circuit:")
print(f"  R_total = {r_series:,.0f} Ω")
print(f"  I       = {i_series*1000:.2f} mA")
print(f"  P       = {p_series*1000:.2f} mW")

# Parallel circuit
r_parallel = resistors_in_parallel(r1, r2, r3)
i_parallel = ohms_law_current(v_supply, r_parallel)
p_parallel = power_dissipation(v_supply, i_parallel)

print(f"\nParallel circuit:")
print(f"  R_total = {r_parallel:.0f} Ω")
print(f"  I       = {i_parallel*1000:.2f} mA")
print(f"  P       = {p_parallel*1000:.2f} mW")

**Figure 7.2** — Circuit analysis using the function library.

In [ ]:
# Extending the library — temperature functions
def celsius_to_fahrenheit(c):
    """Convert Celsius to Fahrenheit."""
    return c * 9 / 5 + 32

def fahrenheit_to_celsius(f):
    """Convert Fahrenheit to Celsius."""
    return (f - 32) * 5 / 9

def celsius_to_kelvin(c):
    """Convert Celsius to Kelvin."""
    return c + 273.15

def kelvin_to_celsius(k):
    """Convert Kelvin to Celsius."""
    return k - 273.15

# Quick demo
temps_c = [0, 20, 37, 100]
for t in temps_c:
    print(f"  {t:6.1f}°C = {celsius_to_fahrenheit(t):7.1f}°F = {celsius_to_kelvin(t):7.2f} K")

**Figure 7.3** — A set of temperature conversion functions with consistent style.

---
## Part 8: Testing Functions with assert

An **assertion** is a sanity check that crashes your program if a condition is `False`:

```python
assert expression, "Error message"
```

Use assertions to:
- Verify function outputs match expected values
- Catch bugs early, *before* they propagate
- Document expected behavior (tests as documentation)

### Testing Tips

| Tip | Example |
|-----|---------|
| Test normal cases | `assert square(3) == 9` |
| Test edge cases | `assert square(0) == 0` |
| Test negative inputs | `assert square(-3) == 9` |
| Use `round()` for floats | `assert round(result, 2) == 3.14` |

In [ ]:
# Basic assertions
def square(x):
    """Return x squared."""
    return x ** 2

# Tests
assert square(0) == 0
assert square(3) == 9
assert square(-4) == 16
assert square(1.5) == 2.25

print("All square() tests passed! ✓")

**Figure 8.1** — Simple assertions testing a `square()` function.

In [ ]:
# Testing with floating point — use round() or abs() tolerance
def circle_area(radius):
    """Return the area of a circle with the given radius."""
    import math
    return math.pi * radius ** 2

# Exact comparison fails for floats:
# assert circle_area(1) == 3.14159265   # might fail!

# ✅ Use round()
assert round(circle_area(1), 4) == 3.1416
assert round(circle_area(0), 4) == 0.0

# ✅ Or use tolerance
result = circle_area(5)
expected = 78.5398
assert abs(result - expected) < 0.001, f"Expected ~{expected}, got {result}"

print("All circle_area() tests passed! ✓")

**Figure 8.2** — Testing floating-point results with `round()` and tolerance.

In [ ]:
# Testing the library functions
assert ohms_law_current(12, 1000) == 0.012
assert round(resistors_in_parallel(1000, 1000), 1) == 500.0
assert resistors_in_series(100, 200, 300) == 600
assert round(power_dissipation(5, 0.002), 4) == 0.01

# Test error handling
try:
    ohms_law_current(12, 0)
    assert False, "Should have raised ValueError"
except ValueError:
    pass  # Expected!

print("All library tests passed! ✓")

**Figure 8.3** — Testing the engineering library, including error cases.

In [ ]:
# The "test first" mindset — write tests before the function
# Suppose Mehmet needs a function to calculate BMI

# Step 1: Write the tests FIRST
def test_bmi():
    assert round(calculate_bmi(70, 1.75), 1) == 22.9   # normal
    assert round(calculate_bmi(100, 1.80), 1) == 30.9   # obese
    assert round(calculate_bmi(50, 1.60), 1) == 19.5    # normal
    print("All BMI tests passed! ✓")

# Step 2: Write the function to make tests pass
def calculate_bmi(weight_kg, height_m):
    """Calculate Body Mass Index.

    Args:
        weight_kg: Body weight in kilograms.
        height_m: Height in meters.

    Returns:
        BMI value (weight / height^2).
    """
    return weight_kg / (height_m ** 2)

# Step 3: Run the tests
test_bmi()

**Figure 8.4** — The test-first approach: write assertions, then implement.

---
## Exercises

Complete each exercise in the code cell below it. Each cell is marked with `# ✏️ [EXn]` so the submission system can find your answers.

**Exercise 1.** Write a function `greet(name, greeting="Merhaba")` that returns a greeting string. Test it with and without the default.

In [ ]:
# ✏️ [EX1]


**Exercise 2.** Write a function `power(base, exponent=2)` that returns `base ** exponent`. Test with `power(5)`, `power(2, 10)`, and `power(3, exponent=3)`.

In [ ]:
# ✏️ [EX2]


**Exercise 3.** Write a function `format_sensor(value, unit="°C", decimals=1)` that returns a formatted string like `"23.5°C"`. Call it using keyword arguments in three different orders.

In [ ]:
# ✏️ [EX3]


**Exercise 4.** Demonstrate the mutable default trap: write a buggy function `collect(item, bag=[])` that appends to a shared list. Show the bug in 3 calls. Then write a fixed version using `None`.

In [ ]:
# ✏️ [EX4]


**Exercise 5.** Write a function `weighted_average(values, weights=None)` where `weights` defaults to equal weights. Use the `None` sentinel pattern. Test with and without weights.

In [ ]:
# ✏️ [EX5]


**Exercise 6.** Write a function `cylinder_volume(radius, height)` with a full Google-style docstring including Args, Returns, and Raises sections. Raise `ValueError` for negative inputs.

In [ ]:
# ✏️ [EX6]


**Exercise 7.** Use `help()` and `__doc__` to inspect your `cylinder_volume` function. Print both outputs.

In [ ]:
# ✏️ [EX7]


**Exercise 8.** Build a small **statistics library** with functions: `mean(data)`, `median(data)`, `std_dev(data)`. Each must have a Google-style docstring.

In [ ]:
# ✏️ [EX8]


**Exercise 9.** Write a function `safe_divide(a, b, default=0.0)` that returns `a/b` or `default` if `b` is zero. Add a docstring and 3 assertions.

In [ ]:
# ✏️ [EX9]


**Exercise 10.** Write a function `clamp(value, low=0, high=100)` that restricts a value to the range [low, high]. Add a docstring and 5 assertions covering below, above, inside, and boundary cases.

In [ ]:
# ✏️ [EX10]


**Exercise 11.** Write `test_first` style: first write 4 assertions for a function `hypotenuse(a, b)` that returns the length of the hypotenuse. Then implement the function. Include a docstring.

In [ ]:
# ✏️ [EX11]


**Exercise 12.** Write a function `describe_list(data, name="data")` that prints the count, mean, min, max of a numeric list, labeled with the `name` parameter. Use keyword arguments when calling.

In [ ]:
# ✏️ [EX12]


**Exercise 13.** Build a mini **unit conversion library** with at least 4 functions (e.g., km↔mi, kg↔lb, L↔gal, m/s↔km/h). Each needs a docstring and 2 assertions.

In [ ]:
# ✏️ [EX13]


**Exercise 14.** Write a function `create_report(title, items, separator="-", width=40)` that returns a formatted multi-line string report. Use all four refactoring skills (Rename, Extract, Simplify, Document).

In [ ]:
# ✏️ [EX14]


**Exercise 15 — Capstone.** Build a tested mini-library for Zeynep’s weather station: `heat_index(temp_c, humidity)`, `wind_chill(temp_c, wind_kmh)`, `classify_weather(temp_c, humidity, wind_kmh)`. Each function needs a full docstring. Write at least 8 assertions total. Use keyword arguments in all test calls.

In [ ]:
# ✏️ [EX15]


---
### 🌉 Bridge to Next Week

This week you mastered advanced function patterns: default arguments, keyword arguments, the mutable default trap, Google-style docstrings, function libraries, and assertion-based testing.

Next week we take the next step: **Modular Thinking — Divide & Conquer**. You will learn to decompose a complex problem into small, independent, testable functions. The goal is to never write a function longer than 10 lines — every complex task becomes a pipeline of simple steps.

---
## 📮 Submission

**STEP 1 —** Fill in your information below and run the cell to verify.

In [ ]:
STUDENT_ID = ""
STUDENT_NAME = ""
STUDENT_EMAIL = ""
CLASS_CODE = ""

import re as _re
_errors = []
if not _re.match(r"^\d{6,12}$", STUDENT_ID): _errors.append("\u274c Student ID must be 6-12 digits")
if len(STUDENT_NAME.strip().split()) < 2: _errors.append("\u274c Enter first and last name")
if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16: _errors.append("\u274c Use your @istun.edu.tr email")
if len(CLASS_CODE.strip()) < 4: _errors.append("\u274c Invalid class code")
if _errors:
    for _e in _errors: print(_e)
    print("\n\u26a0\ufe0f  Fix the errors above and run this cell again.")
else:
    print(f"\u2705 Info OK \u2014 {STUDENT_NAME} ({STUDENT_ID})")
    print(f"   {STUDENT_EMAIL}")
    print(f"\n\U0001f449 Now run the NEXT cell to submit.")

**STEP 2 —** Run the cell below to submit your work. Make sure you have executed all exercise cells first.

In [ ]:
# STEP 2 — Submit your work
import urllib.request, json as _json, re as _re2

_WEEK = "Week_02"
_URL = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"
_SOURCE = "cp2-notebook"

# --- collect answers from executed cells ---
_answers = {}
try:
    _hist = list(In)
except NameError:
    _hist = []

for _i, _c in enumerate(_hist):
    _m = _re2.search(r"#\s*\u270f\ufe0f\s*\[EX(\d+)\]", str(_c))
    if _m:
        _answers[f"ex{_m.group(1)}"] = str(_c)

_payload = _json.dumps({
    "week": _WEEK,
    "source": _SOURCE,
    "studentId": STUDENT_ID,
    "studentName": STUDENT_NAME,
    "studentEmail": STUDENT_EMAIL,
    "classCode": CLASS_CODE,
    "answers": _answers
}).encode()

_req = urllib.request.Request(_URL, data=_payload,
                              headers={"Content-Type": "application/json"})
try:
    _resp = urllib.request.urlopen(_req, timeout=15)
    _body = _json.loads(_resp.read().decode())
    if _body.get("status") == "ok":
        print(f"\u2705 Submitted {len(_answers)} answer(s) for {_WEEK}.")
        print(f"   Timestamp: {_body.get('timestamp', 'n/a')}")
    else:
        print("\u26a0\ufe0f  Server response:", _body)
except Exception as _ex:
    print(f"\u274c Submission failed: {_ex}")
    print("Try again or contact your instructor.")